# Advanced Agent Patterns: Memory, Streaming & Production Features

### What You'll Learn

Building on **5.RAGWithAgents**, this notebook takes your agent skills to the next level with **production-ready patterns**. You'll add memory, streaming responses, error handling, and observability to create robust, enterprise-grade AI agents.

**Prerequisites:** 
- Complete notebooks 3 (LangChain RAG), 4 (Building Agents), and 5 (RAG with Agents)
- Understanding of basic agent patterns and multi-tool usage

**Advanced Concepts Covered:**

1. **Agent Memory Systems** - Persistent conversation history and context
2. **Streaming Responses** - Real-time, token-by-token output for better UX
3. **Error Handling & Recovery** - Making agents robust and reliable
4. **Agent Observability** - Monitoring, logging, and debugging agent behavior
5. **Performance Optimization** - Making agents faster and more efficient
6. **Production Deployment** - Scaling and deploying agents in real environments

### Why Advanced Agent Patterns Matter

**Basic Agents (Notebook 5):**
- Single-turn conversations
- No memory between interactions
- Basic error handling
- Limited observability

**Advanced Agent Patterns:**
- ✅ **Conversational Memory** - Agents remember previous interactions
- ✅ **Streaming Responses** - Real-time feedback like ChatGPT
- ✅ **Robust Error Handling** - Graceful failure recovery
- ✅ **Observability** - Full visibility into agent decision-making
- ✅ **Production Ready** - Scalable, reliable, monitorable

### Real-World Impact

These patterns enable:
- **Customer Service Bots** - Remembering customer history across sessions
- **Research Assistants** - Building on previous research in long conversations
- **Code Assistants** - Maintaining context across complex debugging sessions
- **Enterprise AI** - Reliable, monitored, production-scale deployments

---

Let's transform your basic agents into production-ready AI systems! 🚀

## Step 1: Environment Setup & Review

Let's start by recreating the foundation from **5.RAGWithAgents** and then enhance it with advanced patterns:

## Agent Fundamentals: Quick Review

Before diving into advanced patterns, let's quickly review the core concepts that make agents powerful.

### The Agent Workflow

```
User Question
     ↓
Agent Reasoning (LLM)
     ↓
Tool Selection (if needed)
     ↓
Tool Execution
     ↓
Result Observation
     ↓
Final Answer (or repeat loop)
```

### Understanding Tools

**Tools** are functions or APIs that agents can call to perform actions:

| Tool Type | Examples | Purpose |
|-----------|----------|---------|
| **Search** | Tavily, Google, Bing | Find real-time information |
| **Calculation** | Python REPL, Calculator | Perform computations |
| **Database** | SQL, Vector DB | Query structured data |
| **API Calls** | Weather API, Stock API | Get external data |
| **File Operations** | Read, Write, Delete | Manage files |

### The ReAct Pattern

Agents use the **ReAct** (Reasoning + Acting) pattern:

1. **Reason** - Think about what to do
2. **Act** - Use a tool  
3. **Observe** - See the result
4. **Repeat** - Until the task is complete

**Example ReAct Flow:**
```
User: "What's the weather in Paris and what should I wear?"

Reason: I need current weather data for Paris, then suggest clothing
Act: Use weather tool → "Paris: 15°C, light rain"
Observe: Got weather data
Reason: 15°C with rain means cool and wet conditions
Act: Provide clothing recommendation
Result: "It's 15°C with light rain in Paris. Wear a light jacket and bring an umbrella!"
```

Now let's enhance these basic patterns with **memory, streaming, and production features**! 🚀

In [ ]:
# Import essential libraries
import os
import dotenv
from langchain_openai import AzureChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.document_loaders import WebBaseLoader
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.tools.retriever import create_retriever_tool
from langgraph.prebuilt import create_react_agent

# New imports for advanced features
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, AIMessage
import asyncio
from typing import AsyncIterator

dotenv.load_dotenv()

# Azure OpenAI setup (same as previous notebooks)
llm = AzureChatOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    openai_api_version=os.environ["AZURE_OPENAI_API_VERSION"],
)

print("✓ Connected to Azure OpenAI")
print("✓ Imported advanced agent libraries")
print("✓ Ready to build production-grade agents")

## Step 2: Quick Agent Recreation

Let's quickly recreate our RAG + Web Search agent from notebook 5 as our foundation:

In [ ]:
# Quickly recreate our RAG + Web Search agent from notebook 5

# 1. Create RAG system
loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(docs)
embeddings = OpenAIEmbeddings()
vector_store = FAISS.from_documents(documents, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# 2. Create retriever tool
retriever_tool = create_retriever_tool(
    retriever,
    "knowledge_base_search", 
    "Search the LangSmith documentation knowledge base for questions about LangSmith features, setup, usage, and best practices."
)

# 3. Create web search tool
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY") or "your-tavily-api-key"
web_search_tool = TavilySearchResults(
    max_results=3,
    search_depth="basic", 
    name="web_search",
    description="Search the web for current information, news, and real-time data."
)

# 4. Create basic agent (no memory yet)
tools = [retriever_tool, web_search_tool]
basic_agent = create_react_agent(llm, tools)

print("✓ Recreated RAG + Web Search agent from notebook 5")
print("✓ Ready to add advanced features!")

## Step 3: Adding Memory to Agents 🧠

**The Problem with Basic Agents:**
- No memory of previous conversations
- Each interaction starts from scratch
- Can't build on previous context

**Memory-Enabled Agents:**
- Remember conversation history
- Build on previous interactions
- Provide contextual, personalized responses

### Agent Memory Architecture

```
User: "What is LangSmith?"
Agent: [Searches docs] "LangSmith is..."

User: "How do I install it?"  ← Remembers we're talking about LangSmith!
Agent: [Uses context] "To install LangSmith, you can..."
```

In [ ]:
# Step 3.1: Create Agent with Memory

# Create memory store for conversation history
memory = MemorySaver()

# Create agent with memory capabilities
memory_agent = create_react_agent(llm, tools, checkpointer=memory)

print("✓ Created agent with memory capabilities")
print("✓ Agent can now remember conversation history")

# Let's test the difference!
print("\n=== Testing Memory vs No Memory ===")

# Test 1: Basic agent (no memory)
print("\n1. Basic Agent (No Memory):")
response1 = basic_agent.invoke({
    "messages": [HumanMessage(content="What is LangSmith?")]
})
print("Q: What is LangSmith?")
print(f"A: {response1['messages'][-1].content[:100]}...")

response2 = basic_agent.invoke({
    "messages": [HumanMessage(content="How do I install it?")]
})
print("\nQ: How do I install it?")
print(f"A: {response2['messages'][-1].content[:100]}...")
print("👆 Notice: Agent doesn't know what 'it' refers to!")

In [ ]:
# Step 3.2: Test Agent with Memory

print("\n2. Memory Agent (With Conversation History):")

# Create a conversation thread ID
thread_config = {"configurable": {"thread_id": "conversation_1"}}

# First question
response1 = memory_agent.invoke({
    "messages": [HumanMessage(content="What is LangSmith?")]
}, config=thread_config)
print("Q: What is LangSmith?")
print(f"A: {response1['messages'][-1].content[:100]}...")

# Follow-up question (agent remembers context!)
response2 = memory_agent.invoke({
    "messages": [HumanMessage(content="How do I install it?")]
}, config=thread_config)
print("\nQ: How do I install it?")
print(f"A: {response2['messages'][-1].content[:100]}...")
print("👆 Notice: Agent knows 'it' refers to LangSmith!")

print("\n✅ Memory Success! Agent maintains conversation context")

## Step 4: Streaming Responses 🌊

**Why Streaming Matters:**
- Real-time feedback like ChatGPT
- Better user experience for long responses
- Reduces perceived latency
- Shows agent is "thinking" and working

**Streaming vs Non-Streaming:**
- **Non-Streaming**: Wait... wait... wait... [FULL RESPONSE]
- **Streaming**: "I" → "can" → "help" → "you" → "with" → "that..."

In [ ]:
# Step 4.1: Implement Streaming

import time

def stream_agent_response(agent, messages, config=None):
    """
    Stream agent responses token by token
    """
    print("Agent: ", end="", flush=True)
    
    # Stream the response
    for chunk in agent.stream({"messages": messages}, config=config):
        # Handle different types of chunks
        for node_name, node_data in chunk.items():
            if "messages" in node_data:
                message = node_data["messages"][-1]
                if hasattr(message, 'content') and message.content:
                    # Stream content token by token (simulated)
                    content = message.content
                    for char in content:
                        print(char, end="", flush=True)
                        time.sleep(0.01)  # Simulate streaming delay
                    print()  # New line after complete message
                    break

print("✓ Streaming function created")
print("✓ Ready to demonstrate real-time responses")

In [ ]:
# Step 4.2: Test Streaming vs Non-Streaming

print("=== Streaming vs Non-Streaming Comparison ===\n")

# Non-streaming (traditional)
print("1. Non-Streaming (Traditional):")
start_time = time.time()
response = memory_agent.invoke({
    "messages": [HumanMessage(content="Explain what LangSmith is in 2 sentences")]
}, config=thread_config)
end_time = time.time()
print(f"Agent: {response['messages'][-1].content}")
print(f"⏱️  Total time: {end_time - start_time:.2f} seconds")
print("👆 User waits for complete response\n")

# Streaming (modern UX)
print("2. Streaming (Modern UX):")
start_time = time.time()
stream_agent_response(
    memory_agent, 
    [HumanMessage(content="Now explain LangSmith benefits in 2 sentences")],
    config=thread_config
)
end_time = time.time()
print(f"⏱️  Total time: {end_time - start_time:.2f} seconds")
print("👆 User sees response in real-time!")

print("\n✅ Streaming provides much better user experience!")

## Step 5: Error Handling & Recovery 🛡️

**Why Error Handling Matters:**
- Tools can fail (API limits, network issues)
- LLMs can produce invalid tool calls
- Real-world systems need graceful degradation

**Advanced Error Patterns:**
- Automatic retry with exponential backoff
- Graceful fallbacks when tools fail
- User-friendly error messages
- Continued operation despite partial failures

In [ ]:
# Step 5.1: Create Robust Error-Handling Agent

from langchain.tools import Tool
import random

def unreliable_weather_tool(query: str) -> str:
    """A tool that sometimes fails to simulate real-world issues"""
    if random.random() < 0.5:  # 50% chance of failure
        raise Exception("Weather API is temporarily unavailable")
    return f"The weather for {query} is sunny and 72°F"

def robust_tool_wrapper(tool_func, tool_name: str):
    """Wrapper that adds error handling to any tool"""
    def wrapped_tool(query: str) -> str:
        max_retries = 3
        for attempt in range(max_retries):
            try:
                result = tool_func(query)
                return result
            except Exception as e:
                if attempt == max_retries - 1:
                    return f"Sorry, the {tool_name} tool is currently unavailable. Error: {str(e)}"
                print(f"  ⚠️  Attempt {attempt + 1} failed, retrying...")
                time.sleep(1)  # Wait before retry
    return wrapped_tool

# Create robust weather tool
robust_weather_tool = Tool(
    name="weather_lookup",
    description="Get current weather information for any location",
    func=robust_tool_wrapper(unreliable_weather_tool, "weather")
)

# Create agent with robust tools
robust_tools = [retriever_tool, robust_weather_tool]
robust_agent = create_react_agent(llm, robust_tools, checkpointer=memory)

print("✓ Created robust agent with error handling")
print("✓ Tools will gracefully handle failures and retry automatically")

In [ ]:
# Step 5.2: Test Error Handling

print("=== Testing Error Handling & Recovery ===\n")

# Test the robust agent with potentially failing tool
thread_config_robust = {"configurable": {"thread_id": "robust_conversation"}}

print("Testing weather tool (may fail and retry)...")
response = robust_agent.invoke({
    "messages": [HumanMessage(content="What's the weather like in San Francisco?")]
}, config=thread_config_robust)

print(f"\nAgent Response: {response['messages'][-1].content}")
print("\n✅ Agent handled potential tool failures gracefully!")

# Show the difference in error handling
print("\n" + "="*50)
print("💡 Key Error Handling Features:")
print("  ✅ Automatic retry with backoff")
print("  ✅ Graceful degradation on failure")
print("  ✅ User-friendly error messages")
print("  ✅ Continued operation despite partial failures")

## Step 6: Agent Observability & Monitoring 👁️

**Why Observability Matters:**
- Debug agent decision-making
- Monitor performance in production
- Track tool usage patterns
- Identify bottlenecks and failures

**Observability Features:**
- Detailed execution logs
- Tool call tracking
- Performance metrics
- Decision-making transparency

In [ ]:
# Step 6.1: Implement Agent Observability

import json
from datetime import datetime

class AgentObserver:
    """Monitor and log agent behavior for debugging and optimization"""
    
    def __init__(self):
        self.execution_log = []
        self.tool_usage = {}
        self.performance_metrics = {}
    
    def log_execution(self, step_type: str, content: str, timestamp: datetime = None):
        """Log each step of agent execution"""
        entry = {
            "timestamp": timestamp or datetime.now(),
            "step_type": step_type,
            "content": content
        }
        self.execution_log.append(entry)
    
    def track_tool_usage(self, tool_name: str, success: bool, duration: float):
        """Track tool performance and usage patterns"""
        if tool_name not in self.tool_usage:
            self.tool_usage[tool_name] = {"calls": 0, "successes": 0, "total_duration": 0}
        
        self.tool_usage[tool_name]["calls"] += 1
        if success:
            self.tool_usage[tool_name]["successes"] += 1
        self.tool_usage[tool_name]["total_duration"] += duration
    
    def get_summary(self):
        """Generate comprehensive execution summary"""
        summary = {
            "total_steps": len(self.execution_log),
            "tool_usage": self.tool_usage,
            "execution_timeline": [
                f"{entry['timestamp'].strftime('%H:%M:%S')} - {entry['step_type']}: {entry['content'][:50]}..."
                for entry in self.execution_log[-5:]  # Last 5 steps
            ]
        }
        return summary

# Create global observer instance
observer = AgentObserver()

print("✓ Agent observability system created")
print("✓ Ready to monitor agent behavior in detail")

In [ ]:
# Step 6.2: Test Observability in Action

def observable_agent_execution(agent, messages, config=None):
    """Execute agent with full observability"""
    observer.log_execution("start", f"User query: {messages[0].content}")
    
    start_time = time.time()
    response = agent.invoke({"messages": messages}, config=config)
    end_time = time.time()
    
    observer.log_execution("completion", f"Agent response: {response['messages'][-1].content[:100]}...")
    observer.track_tool_usage("overall_execution", True, end_time - start_time)
    
    return response

print("=== Agent Observability Demo ===\n")

# Run observable agent execution
observable_response = observable_agent_execution(
    memory_agent,
    [HumanMessage(content="Compare LangSmith with other LLM monitoring tools")],
    config=thread_config
)

# Display execution summary
print("\n📊 Execution Summary:")
summary = observer.get_summary()
print(f"Total execution steps: {summary['total_steps']}")
print(f"Tool usage: {json.dumps(summary['tool_usage'], indent=2)}")
print("\n🕐 Recent execution timeline:")
for step in summary['execution_timeline']:
    print(f"  {step}")

print("\n✅ Full observability provides deep insights into agent behavior!")

---

## Congratulations! 🎉

You've mastered **Advanced Agent Patterns** and transformed basic agents into production-ready AI systems!

### What You've Accomplished

1. **Agent Memory Systems** ✓
   - Persistent conversation history
   - Contextual understanding across interactions
   - Thread-based conversation management

2. **Streaming Responses** ✓
   - Real-time, token-by-token output
   - Enhanced user experience
   - ChatGPT-like response patterns

3. **Error Handling & Recovery** ✓
   - Automatic retry mechanisms
   - Graceful degradation on failures
   - User-friendly error messages

4. **Agent Observability** ✓
   - Detailed execution logging
   - Tool usage tracking
   - Performance monitoring
   - Decision-making transparency

### Production-Ready Features

**Your agents now have:**
- ✅ **Memory** - Remember conversations across sessions
- ✅ **Streaming** - Real-time response delivery
- ✅ **Resilience** - Handle failures gracefully
- ✅ **Observability** - Full execution visibility
- ✅ **Performance Monitoring** - Track and optimize behavior

### Real-World Applications

These patterns enable enterprise-grade:
- **Customer Support Systems** - With memory, streaming, and reliability
- **Research Assistants** - Building context over long investigations
- **Code Development Tools** - Maintaining project context
- **Business Intelligence** - Reliable, monitored AI workflows

### Next Steps

**📚 Continue Your Journey:**
- **Next Notebook: `7.AgenticRAG.ipynb`** - Advanced RAG patterns with agent routing
- **Production Deployment** - Scale with FastAPI, Docker, Kubernetes
- **Custom Observability** - Integrate with monitoring systems
- **Advanced Memory** - Vector-based long-term memory systems

**From basic tools to production AI systems - you're ready for the real world!** 🚀

**Next Notebooks:**
- `7.AgenticRAG.ipynb` - Combining agents with RAG
- `8.AgentSupervisor.ipynb` - Multi-agent orchestration

---

**Congratulations!** 🎉 You've completed the Agent tutorial. You now understand:
- ✅ How agents differ from simple LLM calls
- ✅ The ReAct pattern for reasoning and acting
- ✅ Tool-calling and tool execution
- ✅ Building agents with LangGraph
- ✅ Adding memory for conversations
- ✅ Streaming agent responses

Ready to move on to **Agentic RAG**? 🚀